# SAM3 Zero-Shot Detector — Tier 1 Item 3 (the required "genuinely different approach")

Every prior method benchmarked in this project (YOLO26n, the ResNet/
EfficientNet cascade, RT-DETR) is a **supervised detection architecture** —
different networks, but the same paradigm (train/fine-tune a detector on
labeled boxes). SAM3 here is prompted directly with the class name
("person") and never trained on this project's data at all — a genuinely
different paradigm (foundation model + text prompt vs. trained detector),
which is what the assignment specifically asks for beyond architecture
comparisons (`docs/decision_log.md`, 2026-09-12).

**GPU-only, Colab-only** — SAM3 is ViT-based (ultralytics'
`SAM3SemanticPredictor`) and far too heavy for CPU inference, unlike every
other method in this project which could at least run (slowly) on the
M1. This notebook is atomic like `03_gpu_method_comparison.ipynb`: one
step per cell, each calling the same `scripts/*.py` used everywhere else
— no notebook-only logic.

**Note on `sam3_zeroshot` as a module name:** this same SAM3 wrapper
(`src/methods/sam3_zeroshot/model.py::Sam3Detector`) was already used
once before, in `notebooks/01_sam3_autolabel.ipynb`, to *pseudo-label* the
train/val videos — a completely different role (data preparation, not a
benchmarked method). That notebook's own intro explicitly deferred this
exact benchmark to "a separate notebook/script once the gold labels
exist" — this is that notebook.

**Before running:**
- Upload `test_frames.zip` to `/content/drive/MyDrive/object-detection/test_frames.zip`
  (same file used in `03_gpu_method_comparison.ipynb` — skip this step if
  already there from that run).
- You'll need your approved Hugging Face access to `facebook/sam3` (see
  `docs/decision_log.md`, 2026-09-12 — approval came back in minutes last
  time, but request it now if you haven't already).

## Step 0 — Setup: clone the repo, install dependencies, confirm GPU

In [ ]:
import os
if not os.path.exists("object-detection-drone"):
    !git clone https://github.com/Kametor/object-detection-drone.git
%cd object-detection-drone
!git pull origin main --no-edit --no-rebase
!pip install -q -r requirements.txt


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (do not proceed -- SAM3 needs GPU)")


## Step 1 — Get the gold test set's frames

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

TEST_FRAMES_ZIP = "/content/drive/MyDrive/object-detection/test_frames.zip"
!unzip -q -o "{TEST_FRAMES_ZIP}" -d .
!echo "test frames: $(find data/processed/frames/test -name '*.jpg' | wc -l)"


## Step 2 — Hugging Face login and SAM3 weights

Same pattern as `notebooks/01_sam3_autolabel.ipynb`'s cell 6. Requires
your approved access to `facebook/sam3`.

In [ ]:
from huggingface_hub import login, list_repo_files, hf_hub_download

login()  # paste your HF token when prompted

files = list_repo_files("facebook/sam3")
print(files)  # confirm the actual .pt filename before downloading

weight_filename = "sam3.pt"  # adjust if the printed list above shows a different name
sam3_checkpoint_path = hf_hub_download(repo_id="facebook/sam3", filename=weight_filename)
print("Downloaded to:", sam3_checkpoint_path)


## Step 3 — Run SAM3 zero-shot on the gold test set

Prompted with "person" (`configs/34_sam3_zeroshot.yaml`'s `prompt`),
conf=0.25 (unmodified default, matching the same "measure the raw
domain gap" convention used for YOLO26n's and RT-DETR's own zero-shot
checks — see `docs/decision_log.md`), imgsz=1024 (the GPU-memory-safe
value confirmed on a real T4). Patches the checkpoint path in memory
rather than editing the committed config, since the downloaded path is
Colab-session-specific.

**Timing is captured explicitly per frame** (`scripts/34`'s
`per_frame_seconds` list) — SAM3 is the heaviest, slowest model in this
project by a wide margin, and this FPS number is a headline part of the
method comparison's cost section (`CLAUDE.md` Section 7).

In [ ]:
import yaml

config_path = "configs/34_sam3_zeroshot.yaml"
config = yaml.safe_load(open(config_path))
config["checkpoint"] = sam3_checkpoint_path
config["device"] = "cuda"
yaml.safe_dump(config, open("configs/34_T4.yaml", "w"), sort_keys=False)
print(open("configs/34_T4.yaml").read())


In [ ]:
!python scripts/34_sam3_zeroshot_eval.py --config configs/34_T4.yaml


## Step 4 — Evaluate against the gold test set

Same harness as every other method (`scripts/06_evaluate.py`) — mAP,
per-video breakdown, operating point at conf=0.25, all directly
comparable to `yolo26n_zeroshot_T4`, `rtdetr_v2_r18_zeroshot`, etc. in
`results/eval/comparison.csv`.

In [ ]:
!python scripts/06_evaluate.py \
    --predictions results/sam3_zeroshot/predictions.json \
    --method-name sam3_zeroshot_T4 \
    --config configs/06_evaluate.yaml


## Step 5 — Qualitative comparison (GT vs. predictions)

Same visualization used for every other method — sample frames per video,
draw ground truth and SAM3's predictions together.

In [ ]:
!python scripts/07_visualize_eval.py \
    --predictions results/sam3_zeroshot/predictions.json \
    --method-name sam3_zeroshot_T4 \
    --frames-dir data/processed/frames/test \
    --samples-per-video 5 \
    --config configs/06_evaluate.yaml


## Step 6 — Final numbers: accuracy and cost, side by side

`results/eval/comparison.csv` now has a `sam3_zeroshot_T4` row. Print it
next to the other GPU-measured methods for the presentation's headline
table — and the manifest's FPS/timing fields specifically, since that's
the number this run was built to capture.

In [ ]:
import pandas as pd, json, glob

df = pd.read_csv("results/eval/comparison.csv")
display(df[df["method"].isin([
    "yolo26n_zeroshot_T4", "yolo26n_score_fusion_small_stem_T4", "sam3_zeroshot_T4",
])])

manifest_path = sorted(glob.glob("results/manifests/34_sam3_zeroshot_eval_*.json"))[-1]
manifest = json.load(open(manifest_path))
print(f"\nSAM3 timing ({manifest_path}):")
print(f"  mean inference: {manifest['mean_inference_seconds_per_frame']}s/frame -> {manifest['fps']} FPS")
print(f"  min/max frame:  {manifest['min_frame_seconds']}s / {manifest['max_frame_seconds']}s")
print(f"  total wall time for {manifest['num_frames']} frames: {manifest['total_wall_seconds']}s")


## Step 7 — Push results back to GitHub

Text artifacts only — no images, no checkpoints (`sam3.pt` is huge and
access-gated; never committed, stays in the HF cache / re-downloaded via
Step 2 next time).

In [ ]:
import getpass
gh_token = getpass.getpass("GitHub personal access token: ")

!git config user.email "you@example.com"
!git config user.name "Colab"
!git add results/eval results/manifests results/sam3_zeroshot/predictions.json
!git commit -m "Add SAM3 zero-shot benchmark (Tier 1 item 3, T4 GPU)"
!git pull origin main --no-edit --no-rebase
!git push https://{gh_token}@github.com/Kametor/object-detection-drone.git main
